# Auto-Differentiation (Autograd)

In Lesson 07, we explored the elegant mathematics of Backpropagation. We learned how the Chain Rule calculates the exact gradient for every single weight in a network.

However, translating multi-variable calculus into efficient C++ or Python code by hand is a nightmare. In the early days of Deep Learning, Data Scientists spent 80% of their time manually deriving and coding these equations. If you changed your activation function from Sigmoid to ReLU, you had to rewrite all your calculus code from scratch.

Modern frameworks completely eliminate this burden using **Auto-Differentiation** (Autograd). In this lesson, we will look under the hood of PyTorch to see exactly how it dynamically builds a map of your math and solves the calculus for you.

PyTorch’s Autograd engine is a "define-by-run" framework. It does not look at your code and try to solve a static algebraic equation. Instead, as your code executes line-by-line, Autograd silently trails behind you, recording every single mathematical operation into a massive, invisible flowchart.

Let's set up our PyTorch environment to expose this hidden engine.

In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set professional visualization styling
sns.set_theme(style="whitegrid")

print("✅ PyTorch Autograd Engine Environment Ready.")


✅ PyTorch Autograd Engine Environment Ready.


# 1. The Computational Graph (DAG)

When you set `requires_grad=True` on a Tensor (like a neural network weight), you are activating PyTorch's tracking system.

As you perform math operations (addition, multiplication, matrix dots), PyTorch builds a **Directed Acyclic Graph (DAG)** in the background.

* **The Nodes (Leaves)**: The raw Tensors (your weights, biases, and data).
* **The Edges (Roots)**: The actual mathematical functions applied to the tensors.

Let's look at a simple equation: $y = (w \cdot x) + b$

During the **Forward Pass**, PyTorch calculates the output, but it also saves a pointer to a `grad_fn` (Gradient Function).

1. It remembers that $w$ and $x$ were multiplied together, saving a `MulBackward` function.
2. It remembers that $b$ was added to the result, saving an `AddBackward` function.

During the **Backward Pass** (when you call `.backward()`), PyTorch starts at the final output, reads the graph in reverse, and applies the exact calculus derivative for each `grad_fn` using the Chain Rule.

# 2. Leaf Tensors vs. Intermediate Tensors

To manage GPU memory efficiently, PyTorch makes a strict distinction between types of Tensors in the graph.

* **Leaf Tensors**: Tensors created directly by you, the programmer. Your network's Weights and Biases are Leaf Tensors. PyTorch **will** save the `.grad` attribute for these forever, because you need them to update your network.
* **Intermediate Tensors**: Tensors created as the result of a mathematical operation. For example, the hidden layer output ($z$). PyTorch calculates their gradients during the backward pass to keep the Chain Rule flowing, but it **instantly deletes their gradients** the moment the pass is complete to save VRAM.

# 3. The Danger of Gradient Accumulation

In PyTorch, gradients do not overwrite each other; they **accumulate (add together)**.

If you run a forward pass, call `.backward()`, and the gradient for a weight is $5.0$.
If you run *another* forward pass and call `.backward()` again, and the new gradient is $3.0$, PyTorch will simply add them: $5.0 + 3.0 = 8.0$.

By the 100th batch of data, your gradient will be artificially inflated to millions, and your network will violently explode (NaN). This is why calling `optimizer.zero_grad()` or `weight.grad.zero_()` before every single backward pass is a non-negotiable requirement in Deep Learning engineering.

# 4. Exploring the Autograd Engine in Code

Let's build a multi-step mathematical equation. We will inspect the hidden `grad_fn` pointers, run the backward pass, and prove that intermediate tensors have their gradients deleted.

In [2]:
# 1. Define Leaf Tensors (The parameters we want to optimize)
w = torch.tensor([2.0], requires_grad=True)
b = torch.tensor([-1.0], requires_grad=True)
x = torch.tensor([5.0]) # The input data (does not require gradients)

print("--- 1. Initial State (Leaf Tensors) ---")
print(f"Is 'w' a leaf? {w.is_leaf}")
print(f"Is 'x' a leaf? {x.is_leaf}")

# 2. The Forward Pass (Building the Graph)
# Operation 1: Multiplication
mult_step = w * x
# Operation 2: Addition
z = mult_step + b
# Operation 3: Activation (ReLU)
a = torch.relu(z)

print("\n--- 2. Inspecting the Computational Graph (grad_fn) ---")
print(f"The operation that created 'mult_step': {mult_step.grad_fn}")
print(f"The operation that created 'z':         {z.grad_fn}")
print(f"The operation that created 'a':         {a.grad_fn}")
print("Insight: PyTorch has secretly mapped the entire calculus chain!")

# 3. The Backward Pass
a.backward()

print("\n--- 3. Gradients Calculated ---")
print(f"Gradient of w (dL/dw): {w.grad.item()}")
print(f"Gradient of b (dL/db): {b.grad.item()}")

# 4. Checking Intermediate Tensors
# PyTorch deletes the gradients of intermediate steps to save memory.
# If we try to look at mult_step.grad, it will be None.
print(f"Gradient of intermediate step 'z': {z.grad}")
print("Insight: 'z' was an intermediate tensor. Its gradient was used for the Chain Rule, then instantly thrown into the trash to free up GPU VRAM.")

--- 1. Initial State (Leaf Tensors) ---
Is 'w' a leaf? True
Is 'x' a leaf? True

--- 2. Inspecting the Computational Graph (grad_fn) ---
The operation that created 'mult_step': <MulBackward0 object at 0x724b1c9fd360>
The operation that created 'z':         <AddBackward0 object at 0x724b1c9fd330>
The operation that created 'a':         <ReluBackward0 object at 0x724b1c9fd360>
Insight: PyTorch has secretly mapped the entire calculus chain!

--- 3. Gradients Calculated ---
Gradient of w (dL/dw): 5.0
Gradient of b (dL/db): 1.0
Gradient of intermediate step 'z': None
Insight: 'z' was an intermediate tensor. Its gradient was used for the Chain Rule, then instantly thrown into the trash to free up GPU VRAM.


/tmp/ipykernel_3895/3740808420.py:34: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at aten/src/ATen/core/TensorBody.h:489.)
  print(f"Gradient of intermediate step 'z': {z.grad}")


# 5. Context Managers: `torch.no_grad()`

The Autograd engine is brilliant, but building the computational graph consumes massive amounts of RAM and slows down processing.

When you are simply using your model to make a prediction in production (Inference), or when you are manually updating your weights, you **do not want** PyTorch to track the math.

We wrap these operations in a context manager called `torch.no_grad()`. This temporarily unplugs the Autograd engine, making your code run incredibly fast and consume minimal memory.

In [3]:
print("\n--- 4. Memory Management with no_grad() ---")

# Without no_grad: PyTorch tracks the operation
y_tracked = w * 10
print(f"Tracked Operation: requires_grad = {y_tracked.requires_grad}")

# With no_grad: PyTorch ignores the operation
with torch.no_grad():
    y_untracked = w * 10
    
print(f"Untracked Operation: requires_grad = {y_untracked.requires_grad}")
print("Insight: Always use torch.no_grad() when updating weights or making final predictions in production!")


--- 4. Memory Management with no_grad() ---
Tracked Operation: requires_grad = True
Untracked Operation: requires_grad = False
Insight: Always use torch.no_grad() when updating weights or making final predictions in production!


## Real-World Use Case or Analogy:

Think of Auto-Differentiation like **An Expense Tracking System on a Corporate Trip**:

* **Manual Calculus**: At the end of a 5-day trip, the accountant hands you a massive, blank spreadsheet. You have to sit down, remember every single coffee, taxi, and hotel you bought, find the receipts, and manually reconstruct the entire total cost from memory.
* **The Autograd Engine (The Corporate Credit Card)**: Every single time you swipe the card (Forward Pass), the bank automatically creates a digital timestamp, categorizes the vendor, and saves the receipt in the cloud (Building the Computational Graph).
* **The Backward Pass**: At the end of the trip, you don't do any math. You just click a single button (`.backward()`), and the system instantly traverses the entire digital ledger in reverse, generating a flawless, itemized breakdown of exactly how much each category contributed to the final bill (The Gradients).